In [56]:
# Зареждане на набора от данни
from torch_geometric.datasets import Planetoid

dataset = Planetoid(root="data/Cora", name="Cora")
data = dataset[0]

In [90]:
# Конструиране на Residual GraphSAGE модела с Mean Aggregation
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv


class ResidualGraphSAGE(torch.nn.Module):

    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()

        self.conv1 = SAGEConv(
            in_channels,
            hidden_channels,
            aggr="mean"
        )

        self.conv2 = SAGEConv(
            hidden_channels,
            hidden_channels,
            aggr="mean"
        )

        self.classifier = torch.nn.Linear(
            hidden_channels,
            out_channels
        )

        self.dropout = 0.5

    def forward(self, x, edge_index):

        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = F.dropout(
            h,
            p=self.dropout,
            training=self.training
        )

        h_new = self.conv2(h, edge_index)

        # Residual връзка
        h = h + h_new

        h = F.relu(h)

        out = self.classifier(h)

        return out

In [91]:
# Създаване на модела
model = ResidualGraphSAGE(
    dataset.num_features,
    64,
    dataset.num_classes)

In [82]:
# Конструиране на Jumping Knowledge Network
import torch
import torch.nn.functional as F
from torch_geometric.nn import (SAGEConv, JumpingKnowledge)

class JKGraphSAGE(torch.nn.Module):

    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()

        self.conv1 = SAGEConv(
            in_channels,
            hidden_channels,
            aggr="mean" )

        self.conv2 = SAGEConv(
            hidden_channels,
            hidden_channels,
            aggr="mean" )

        self.conv3 = SAGEConv(
            hidden_channels,
            hidden_channels,
            aggr="mean"  )

        self.jump = JumpingKnowledge(mode="cat") # или mode="max"

        self.classifier = torch.nn.Linear(
            hidden_channels * 3, # не се умножава по 3 при mode="max"
            out_channels )

        self.dropout = 0.5

    def forward(self, x, edge_index):

        h1 = self.conv1(x, edge_index)
        h1 = F.relu(h1)
        h1 = F.dropout(
            h1,
            p=self.dropout,
            training=self.training )

        h2 = self.conv2(h1, edge_index)
        h2 = F.relu(h2)
        h2 = F.dropout(
            h2,
            p=self.dropout,
            training=self.training )

        h3 = self.conv3(h2, edge_index)
        h3 = F.relu(h3)

        h = self.jump([h1, h2, h3])

        out = self.classifier(h)

        return out

In [83]:
# Създаване на модела
model = JKGraphSAGE(
    dataset.num_features,
    64,
    dataset.num_classes)

In [92]:
# Инициализиране на оптимизатора
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01)

In [93]:
# Функция за обучение
import torch.nn.functional as F

def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)

    loss = F.cross_entropy(
        out[data.train_mask],
        data.y[data.train_mask]    )

    loss.backward()
    optimizer.step()

    return loss.item()

In [94]:
# Обучение на модела и измерване на времето
import time

start_time = time.time()

for epoch in range(1, 201):

    loss = train()

    if epoch % 20 == 0:
        print(
            f"Epoch: {epoch:3d}, "
            f"Loss: {loss:.4f}"
        )

training_time = time.time() - start_time

print(f"\nВреме за обучение: {training_time:.2f} s")

Epoch:  20, Loss: 0.0000
Epoch:  40, Loss: 0.0006
Epoch:  60, Loss: 0.0006
Epoch:  80, Loss: 0.0000
Epoch: 100, Loss: 0.0002
Epoch: 120, Loss: 0.0000
Epoch: 140, Loss: 0.0001
Epoch: 160, Loss: 0.0000
Epoch: 180, Loss: 0.0000
Epoch: 200, Loss: 0.0000

Време за обучение: 11.27 s


In [95]:
# Оценяване
from sklearn.metrics import accuracy_score, f1_score
def evaluate():

    model.eval()

    with torch.no_grad():

        out = model(data.x, data.edge_index)
        pred = out.argmax(dim=1)

    y_true = data.y[data.test_mask].cpu()
    y_pred = pred[data.test_mask].cpu()

    accuracy = accuracy_score(y_true, y_pred)

    f1 = f1_score(
        y_true,
        y_pred,
        average="macro"
    )

    return accuracy, f1

In [96]:
# Извеждане на резултатите
accuracy, f1 = evaluate()

print(f"Accuracy      : {accuracy:.4f}")
print(f"Macro F1-score: {f1:.4f}")
print(f"Параметри     : {num_params}")
print(f"Време         : {training_time:.2f} s")

Accuracy      : 0.7250
Macro F1-score: 0.7211
Параметри     : 201351
Време         : 11.27 s


In [97]:
# Изчисляване на броя параметри
num_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad)

print(f"Брой параметри: {num_params}")

Брой параметри: 192199


In [98]:
model

ResidualGraphSAGE(
  (conv1): SAGEConv(1433, 64, aggr=mean)
  (conv2): SAGEConv(64, 64, aggr=mean)
  (classifier): Linear(in_features=64, out_features=7, bias=True)
)